In [2]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import kornia.augmentation as K

In [ ]:
def load_patch_dict_as_tensor(Patch_dict):
    
    # try first as a for loop and then see if can switch to comprehension

    band_names = ["B2", "B3", "B4", "B5", "B6"]

    feature_list = Patch_dict["features"]

    patch_list = []
    label_list = []
    location_list = []

    for feat in feature_list:

        band_arrays = [np.array(feat["properties"][band]) for band in band_names]
        stacked_patch = np.stack(band_arrays) # (C,H,W)

        patch_list.append(stacked_patch)
        label_list.append(feat["properties"]["lc"])
        location_list.append(feat["properties"]["location"])

    patches = np.stack(patch_list) # (N, C, H, W)
    patches = torch.from_numpy(patches).float()

    labels= torch.tensor(label_list).long()
    locations = np.array(location_list) # array for easier boolean masking but no need for tensor


    return patches, labels, locations

In [ ]:
# Version using xomprehension

def load_patch_dict_as_tensor(Patch_dict, band_list = None):
    if not band_list:
        band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

    feature_list = Patch_dict["features"]

    patches = np.stack([
        np.stack([np.array(F["properties"][band]) for band in band_list]) 
        for F in feature_list 
    ])

    labels = torch.tensor([F["properties"]["label"] for F in feature_list]).long()

    locations = np.array([F["properties"]["location"] for F in feature_list]) # array for easier boolean masking but no need for tensor

    patches = torch.from_numpy(patches).float()

    patches = patches.permute(0,3,1,2) # changes stacked numpy (N,H,W,C) to pytorch (N,C,H,W)

    return patches, labels, locations

patches, labels, locations = load_patch_dict_as_tensor(Patches_dict, band_list= None)

# check the shape

print(patches.shape)


In [ ]:
# next need to calcualte the mean and std for just the train region, can create a test_mask and val_mask and then train_masl = ~test_mask & ~val_mask

test_region = "Valsequillo"
val_region = "Vembanad"

test_mask = locations == test_region
val_mask = locations == val_region

train_mask = ~test_mask & ~ val_mask

train_patches, train_labels = patches[train_mask], labels[train_mask]
val_patches, val_labels = patches[val_mask], labels[val_mask]
test_patches, test_labels = patches[test_mask], labels[test_mask]


mean = train_patches.mean(dim= (0,2,3)) # will average over the 0,2,3 axes and keep the 1 axis (channels / bands) separate output shape (C,)
std = train_patches.std(dim = (0,2,3))

mean = torch.tensor(mean).view(1,-1,1,1) # changes mean from (C,) to (1,C,1,1)
std = torch.tensor(std).view(1,-1,1,1)

# standardise all the patches using the mean and std from the train patches - np maybe do this for each patch in the dataset in __getitem__?

# std_train_patches = (train_patches - mean)/std
# std_val_patches = (val_patches-mean)/std
# std_test_patches = (test_patches-mean)/ std




In [ ]:

class PatchDataset(Dataset):
    def __init__(self, patches, labels, mean, std):
        
        mean = torch.tensor(mean).view(1,-1,1,1) # changes from (C,) to (1,C,1,1)
        std = torch.tensor(std).view(1,-1,1,1)
    
        self.patches = (patches - mean)/std 
        self.labels = labels 

    def __len__(self):
        return len(patches)

    def __getitem__(self, idx):
        return self.patches[idx], self.labels[idx]
    

    
patch_dataset = PatchDataset(patches = patches, mean = mean, std = std)
loader = DataLoader(
    dataset= patch_dataset,
    shuffle= True,
    batch_size= 32,
    num_workers= 4,
    pin_memory= True
    )
    
aug = K.AugmentationSequential([
    K.RandomRotation(degrees = 90.0, p =0.5),
    K.RandomHorizontalFlip(p= 0.25),
    K.RandomVerticalFlip(p= 0.25)
])

for batch_X, batch_y in loader:

